# Fine-tuning de DETR para detección de objetos

Ajustaremos DETR preentrenado en COCO para detectar una sola clase: globos.

**Conceptos:** detección basada en conjuntos, object queries, formato COCO, cajas normalizadas, padding, fine-tuning y postprocesamiento.

**Resultado esperado:** primero se observan detecciones del modelo COCO; después un entrenamiento breve verifica que datos, pérdidas, gradientes y guardado funcionan. Dos épocas son un smoke test, no evidencia de convergencia.


## 1. Instalación y reproducibilidad

Usamos la implementación mantenida en Transformers. Registramos versiones y semilla. timm provee el backbone convolucional en instalaciones que lo requieren.


In [ ]:
!pip install -q transformers timm


In [ ]:
import copy
import json
import random
import time
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

import matplotlib.pyplot as plt
import numpy as np
import requests
import torch
import transformers
from matplotlib.patches import Rectangle
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from transformers import AutoConfig, AutoImageProcessor, DetrForObjectDetection

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)
print("Dispositivo:", device)


## 2. Inferencia con DETR preentrenado

DETR emite un conjunto fijo de object queries. Cada query predice una clase, incluida no-object, y una caja normalizada en formato centro_x, centro_y, ancho, alto.

AutoImageProcessor aplica normalización y redimensionamiento; post_process_object_detection filtra no-object, aplica el umbral y convierte cajas a píxeles xmin, ymin, xmax, ymax.

**Resultado esperado:** cajas sobre objetos COCO de la imagen. Bajar el umbral conserva más queries.


In [ ]:
MODEL_CHECKPOINT = "facebook/detr-resnet-50"

image_processor = AutoImageProcessor.from_pretrained(
    MODEL_CHECKPOINT,
    size={"shortest_edge": 480, "longest_edge": 640},
)
pretrained_model = DetrForObjectDetection.from_pretrained(
    MODEL_CHECKPOINT
).to(device)
pretrained_model.eval()

sample_url = (
    "http://images.cocodataset.org/val2017/000000039769.jpg"
)
response = requests.get(sample_url, timeout=30)
response.raise_for_status()
sample_image_path = Path("detr_demo.jpg")
sample_image_path.write_bytes(response.content)
sample_image = Image.open(sample_image_path).convert("RGB")

inputs = image_processor(
    images=sample_image,
    return_tensors="pt",
).to(device)
with torch.inference_mode():
    outputs = pretrained_model(**inputs)

target_sizes = torch.tensor(
    [sample_image.size[::-1]],
    device=device,
)
detections = image_processor.post_process_object_detection(
    outputs,
    threshold=0.8,
    target_sizes=target_sizes,
)[0]

print("Queries del modelo:", outputs.logits.shape[1])
print("Detecciones conservadas:", len(detections["scores"]))


La función siguiente dibuja cajas en formato xmin, ymin, xmax, ymax. Recibe las etiquetas ya convertidas a texto para no depender de variables globales.


In [ ]:
def show_detections(image, boxes, labels, scores=None, title=""):
    # Dibuja cada caja y su etiqueta sobre una copia visual de la imagen.
    figure, axis = plt.subplots(figsize=(11, 8))
    axis.imshow(image)

    for detection_index, (box, label) in enumerate(
        zip(boxes, labels)
    ):
        xmin, ymin, xmax, ymax = box
        axis.add_patch(
            Rectangle(
                (xmin, ymin),
                xmax - xmin,
                ymax - ymin,
                fill=False,
                edgecolor="lime",
                linewidth=2,
            )
        )
        text = label
        if scores is not None:
            text += f" {scores[detection_index]:.2f}"
        axis.text(
            xmin,
            max(0, ymin - 4),
            text,
            color="black",
            bbox={"facecolor": "lime", "alpha": 0.7},
        )

    axis.set_title(title)
    axis.axis("off")
    plt.show()


predicted_class_names = [
    pretrained_model.config.id2label[class_id.item()]
    for class_id in detections["labels"]
]
show_detections(
    sample_image,
    detections["boxes"].detach().cpu().tolist(),
    predicted_class_names,
    detections["scores"].detach().cpu().tolist(),
    title="DETR preentrenado en COCO",
)


## 3. Dataset Balloon

Balloon tiene polígonos anotados en formato VIA. Lo descargamos una vez y calculamos una caja envolvente por polígono.

El procesador de DETR espera anotaciones de detección estilo COCO:

- image_id identifica la imagen.
- bbox usa xmin, ymin, ancho, alto en píxeles.
- category_id es un entero por clase.
- area e iscrowd completan el objeto COCO.

**Resultado esperado:** carpetas train y val con imágenes y via_region_data.json.


In [ ]:
dataset_root = Path("data/balloon")
train_annotations_path = dataset_root / "train/via_region_data.json"
validation_annotations_path = dataset_root / "val/via_region_data.json"

if not (
    train_annotations_path.exists()
    and validation_annotations_path.exists()
):
    archive_path = Path("balloon_dataset.zip")
    if not archive_path.exists():
        urlretrieve(
            (
                "https://github.com/matterport/Mask_RCNN/releases/"
                "download/v2.1/balloon_dataset.zip"
            ),
            archive_path,
        )
    Path("data").mkdir(exist_ok=True)
    with ZipFile(archive_path) as archive:
        archive.extractall("data")

if not train_annotations_path.exists():
    raise FileNotFoundError(
        f"No se encontró {train_annotations_path}."
    )

print("Train:", train_annotations_path)
print("Validación:", validation_annotations_path)


BalloonDataset mantiene explícita la conversión de cada polígono. El procesador transforma la caja a la representación normalizada que usa DETR.


In [ ]:
class BalloonDataset(Dataset):
    def __init__(self, image_directory, annotations_path, processor):
        self.image_directory = Path(image_directory)
        self.processor = processor

        with Path(annotations_path).open(encoding="utf-8") as file:
            annotation_dictionary = json.load(file)

        self.records = [
            record
            for record in annotation_dictionary.values()
            if (self.image_directory / record["filename"]).exists()
        ]

    def __len__(self):
        return len(self.records)

    def _objects_from_record(self, record):
        regions = record.get("regions", [])
        if isinstance(regions, dict):
            regions = regions.values()

        objects = []
        for region in regions:
            shape = region["shape_attributes"]
            x_coordinates = shape["all_points_x"]
            y_coordinates = shape["all_points_y"]

            xmin = float(min(x_coordinates))
            ymin = float(min(y_coordinates))
            width = float(max(x_coordinates) - xmin)
            height = float(max(y_coordinates) - ymin)

            objects.append(
                {
                    "bbox": [xmin, ymin, width, height],
                    "category_id": 0,
                    "area": width * height,
                    "iscrowd": 0,
                }
            )
        return objects

    def __getitem__(self, index):
        record = self.records[index]
        image_path = self.image_directory / record["filename"]
        image = Image.open(image_path).convert("RGB")

        coco_target = {
            "image_id": index,
            "annotations": self._objects_from_record(record),
        }
        # El procesamiento se hace por batch en collate_fn para
        # aplicar padding consistente a imágenes de distinto tamaño.
        return {"image": image, "target": coco_target}

    def raw_sample(self, index):
        # Entrega imagen y cajas originales para una inspección visual.
        record = self.records[index]
        image = Image.open(
            self.image_directory / record["filename"]
        ).convert("RGB")
        objects = self._objects_from_record(record)
        boxes_xyxy = []
        for object_annotation in objects:
            xmin, ymin, width, height = object_annotation["bbox"]
            boxes_xyxy.append(
                [xmin, ymin, xmin + width, ymin + height]
            )
        return image, boxes_xyxy


train_dataset = BalloonDataset(
    dataset_root / "train",
    train_annotations_path,
    image_processor,
)
validation_dataset = BalloonDataset(
    dataset_root / "val",
    validation_annotations_path,
    image_processor,
)

print("Imágenes de train:", len(train_dataset))
print("Imágenes de validación:", len(validation_dataset))


Inspeccionamos una anotación antes de entrenar. Este paso detecta errores de formato mucho antes que una métrica.

**Resultado esperado:** cada caja verde rodea un globo.


In [ ]:
raw_image, ground_truth_boxes = train_dataset.raw_sample(0)
show_detections(
    raw_image,
    ground_truth_boxes,
    ["globo"] * len(ground_truth_boxes),
    title="Anotaciones originales de Balloon",
)


## 4. Padding y DataLoader

Las imágenes tienen tamaños diferentes. collate_fn procesa el batch completo y agrega padding hasta su mayor alto y ancho y produce pixel_mask para indicar qué píxeles son reales.

**Resultado esperado:** pixel_values tiene forma (batch, 3, alto, ancho), mientras labels es una lista porque cada imagen puede contener distinto número de objetos.


In [ ]:
def collate_fn(batch):
    # El procesador transforma y completa todas las imágenes del batch juntas.
    encoded_batch = image_processor(
        images=[item["image"] for item in batch],
        annotations=[item["target"] for item in batch],
        return_tensors="pt",
    )
    return {
        "pixel_values": encoded_batch["pixel_values"],
        "pixel_mask": encoded_batch["pixel_mask"],
        "labels": encoded_batch["labels"],
    }



BATCH_SIZE = 2
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn,
    generator=torch.Generator().manual_seed(SEED),
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn,
)

example_batch = next(iter(train_loader))
print("pixel_values:", example_batch["pixel_values"].shape)
print("pixel_mask:", example_batch["pixel_mask"].shape)
print("Objetos por imagen:", [
    len(labels["class_labels"])
    for labels in example_batch["labels"]
])


## 5. Modelo para una clase

Reutilizamos backbone y Transformer preentrenados, pero reemplazamos la cabeza de clases COCO por una salida para globo. ignore_mismatched_sizes autoriza ese reemplazo.

Para que el ejemplo sea más liviano congelamos parámetros cuyo nombre contiene backbone. La cabeza y el Transformer siguen entrenables.


In [ ]:
id_to_label = {0: "globo"}
label_to_id = {"globo": 0}

detr_config = AutoConfig.from_pretrained(MODEL_CHECKPOINT)
detr_config.id2label = id_to_label
detr_config.label2id = label_to_id
detr_config.num_labels = 1

model = DetrForObjectDetection.from_pretrained(
    MODEL_CHECKPOINT,
    config=detr_config,
    ignore_mismatched_sizes=True,
)

for parameter_name, parameter in model.named_parameters():
    if "backbone" in parameter_name:
        parameter.requires_grad = False

model = model.to(device)
trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)
print(f"Parámetros entrenables: {trainable_parameters:,}")


## 6. Entrenamiento breve

Cuando se entregan labels, DETR calcula clasificación, error L1 de cajas y generalized IoU después de asignar predicciones a objetos mediante matching bipartito.

La función usa gradientes solo en train y reporta pérdida media por imagen.

**Resultado esperado:** el código completa forward y backward sin errores. Dos épocas no garantizan una curva suave.


In [ ]:
def run_detr_epoch(model, data_loader, device, optimizer=None):
    is_training = optimizer is not None
    model.train(mode=is_training)

    total_loss = 0.0
    total_images = 0

    for batch in data_loader:
        pixel_values = batch["pixel_values"].to(device)
        pixel_mask = batch["pixel_mask"].to(device)
        labels = [
            {
                key: value.to(device)
                for key, value in image_labels.items()
            }
            for image_labels in batch["labels"]
        ]

        if is_training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_training):
            outputs = model(
                pixel_values=pixel_values,
                pixel_mask=pixel_mask,
                labels=labels,
            )
            loss = outputs.loss
            if is_training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=0.1,
                )
                optimizer.step()

        batch_size = pixel_values.size(0)
        total_loss += loss.item() * batch_size
        total_images += batch_size

    return total_loss / total_images


In [ ]:
EPOCHS = 2
LEARNING_RATE = 1e-5
optimizer = torch.optim.AdamW(
    (
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad
    ),
    lr=LEARNING_RATE,
    weight_decay=1e-4,
)

best_validation_loss = float("inf")
best_state = copy.deepcopy(model.state_dict())
history = {"train_loss": [], "validation_loss": []}

for epoch in range(1, EPOCHS + 1):
    start_time = time.time()
    train_loss = run_detr_epoch(
        model,
        train_loader,
        device,
        optimizer,
    )
    validation_loss = run_detr_epoch(
        model,
        validation_loader,
        device,
    )

    history["train_loss"].append(train_loss)
    history["validation_loss"].append(validation_loss)
    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        best_state = copy.deepcopy(model.state_dict())

    print(
        f"Época {epoch}/{EPOCHS} | "
        f"train loss {train_loss:.4f} | "
        f"val loss {validation_loss:.4f} | "
        f"{time.time() - start_time:.1f} s"
    )

model.load_state_dict(best_state)

output_directory = Path("outputs/detr_balloon")
output_directory.mkdir(parents=True, exist_ok=True)
model.save_pretrained(output_directory)
image_processor.save_pretrained(output_directory)

experiment_config = {
    "transformers_version": transformers.__version__,
    "checkpoint": MODEL_CHECKPOINT,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "seed": SEED,
    "processor_size": {
        "shortest_edge": 480,
        "longest_edge": 640,
    },
}
with (output_directory / "course_experiment.json").open(
    "w",
    encoding="utf-8",
) as config_file:
    json.dump(experiment_config, config_file, indent=2)

print("Modelo y configuración guardados en:", output_directory)


## 7. Predicción cualitativa

Visualizamos ground truth y predicción en una imagen de validación. El umbral 0.5 es una decisión de visualización, no un hiperparámetro aprendido.

**Resultado esperado:** tras un smoke test puede haber cajas faltantes o imprecisas. Eso no es un error de ejecución; se necesitarían más épocas y evaluación mAP para juzgar el modelo.


In [ ]:
validation_image, validation_boxes = validation_dataset.raw_sample(0)

prediction_inputs = image_processor(
    images=validation_image,
    return_tensors="pt",
).to(device)
model.eval()
with torch.inference_mode():
    prediction_outputs = model(**prediction_inputs)

target_sizes = torch.tensor(
    [validation_image.size[::-1]],
    device=device,
)
prediction = image_processor.post_process_object_detection(
    prediction_outputs,
    threshold=0.5,
    target_sizes=target_sizes,
)[0]

show_detections(
    validation_image,
    validation_boxes,
    ["globo"] * len(validation_boxes),
    title="Ground truth de validación",
)
show_detections(
    validation_image,
    prediction["boxes"].detach().cpu().tolist(),
    ["globo"] * len(prediction["boxes"]),
    prediction["scores"].detach().cpu().tolist(),
    title="Predicción de DETR ajustado",
)


## 8. Ejercicios

**Ejercicio 1 — Umbral.** Compare 0.2, 0.5 y 0.8 en la misma imagen.

**Resultado esperado:** un umbral alto reduce cajas, con posible pérdida de objetos reales.

**Ejercicio 2 — Congelamiento.** Descongele el backbone y use un learning rate menor para él mediante dos grupos del optimizador.

**Resultado esperado:** aumenta memoria y tiempo; cualquier mejora debe medirse con varias ejecuciones.

**Ejercicio 3 — Evaluación.** Implemente AP@50 sobre todo val o conecte un evaluador COCO. Defina IoU, orden por confianza y manejo de detecciones duplicadas.

**Checklist:** cajas originales inspeccionadas, class id desde cero, processor consistente, test no usado para ajustar y configuración guardada.
